# Python modules, packages, `__name__`, and `__main__`

This guide covers the four topics in `assignment.txt`. Read from top to bottom and run each code cell with the Run button or Shift+Enter. A Python kernel is needed; all examples use only Python's standard library.

**Big picture:** a module is reusable Python code; a package groups modules; `__name__` identifies how a module was loaded; and `"__main__"` identifies the entry point of a Python program.

## 1. Modules

A **module** is a unit of Python code that you can import. Often it is a `.py` file. It can contain functions, variables, classes, and other statements.

For example, `math` is a standard-library module. `import math` gives you access to its contents through the name `math`:

In [1]:
import math

print(math.sqrt(25))  # Call a function from the module
print(math.pi)        # Access a variable from the module

5.0
3.141592653589793


You can also import a particular name or give a module a shorter alias:

In [2]:
from math import sqrt
import math as m

print(sqrt(36))
print(m.sqrt(49))

6.0
7.0


### Your existing module

Your `test/utils.py` is a module containing `BATCH` and `add`. In `test/main.py`, `import utils` lets you use `utils.BATCH` and `utils.add(12, 23)`.

When a module is imported for the first time in a process, Python executes its top-level statements and normally keeps the module in `sys.modules`. A later ordinary import normally reuses it instead of running it again. The statements inside a function do not run just because the function is defined; they run when it is called.

## 2. Packages

A **package** is a way to organize related modules under a shared name. A typical package has a folder with an `__init__.py` file:

```text
my_project/
    shop/
        __init__.py
        prices.py
        orders.py
```

Here, `shop` is the package and `shop.prices` is a module inside it. The dot means “inside,” not a filename separator you need to type in a file path. For example, `import shop.prices` makes `shop.prices` available. Python must be able to find `my_project` on its import path.

`__init__.py` can be empty, or it can contain package initialization code that runs when the package is first imported. Some modern **namespace packages** can omit it, but using it is a straightforward starting point.

A real example is the standard-library `email` package and its `email.message` module:

In [ ]:
import email.message

message = email.message.EmailMessage()
message['Subject'] = 'Hello'
print(message['Subject'])
print('Package name:', email.__name__)
print('Module name:', email.message.__name__)

## 3. `__name__`

`__name__` is a special variable that Python sets in each module. Its value tells you the name under which that module is running.

| How the code is used | Value of `__name__` inside that code |
| --- | --- |
| Run `python utils.py` directly | `"__main__"` |
| Import it with `import utils` | `"utils"` |
| Import a module with `import shop.prices` | `"shop.prices"` |

The notebook itself typically executes cells with `__name__` set to `"__main__"`. That does **not** mean you ran a `.py` file directly; the notebook kernel is running the cells as its main namespace.

In [3]:
print('This notebook cell:', __name__)
print('Imported math module:', math.__name__)

This notebook cell: __main__
Imported math module: math


## 4. `"__main__"` and the main guard

`"__main__"` is a **string value**, not a function. Python uses it as `__name__` for the program's entry point. This condition is often called the **main guard**:

```python
if __name__ == "__main__":
    print("Run only when this file is the entry point")
```

When the file is run directly, the condition is true. When it is imported under its module name, the condition is false. Statements outside the guard still run on import.

In your `test/utils.py`, the `BATCH` assignment and `add` definition are outside the guard, so they are available when imported. The test print and `add(1, 2)` are inside the guard, so a normal `import utils` does not run them. The guard in `test/main.py` runs its calls when you run `main.py` directly.

### See direct execution versus import

This example creates a small module in a temporary directory and runs it two ways. Notice that the top-level print always runs, but the print inside `if __name__ == "__main__":` runs only for direct execution.

In [6]:
from pathlib import Path
import subprocess
import sys
from tempfile import TemporaryDirectory

with TemporaryDirectory() as folder:
    demo = Path(folder) / 'study_demo.py'
    demo.write_text('''print("Top level; __name__ =", __name__)\n
if __name__ == "__main__":
    print("This runs only when the file is run directly")
''', encoding='utf-8')

    print('--- Run the file directly ---')
    direct = subprocess.run([sys.executable, str(demo)], capture_output=True, text=True)
    print(direct.stdout, end='')

    print('--- Import the file ---')
    imported = subprocess.run(
        [sys.executable, '-c', 'import study_demo'],
        cwd=folder,
        capture_output=True,
        text=True,
    )
    print(imported.stdout, end='')

--- Run the file directly ---
Top level; __name__ = __main__
This runs only when the file is run directly
--- Import the file ---
Top level; __name__ = study_demo


In the first run, both the top-level print and the guarded print execute. In the import, the top-level print executes, but the guarded print does not.

The main guard also works when a module is run with `python -m package.module`: that module runs as `"__main__"`. This is different from `__main__.py`, a special filename that can provide an entry point when you run `python -m package`.

## Practice

1. In `test/utils.py`, which line would run when the file is imported: `BATCH = "FDE1.0"` or `add(1, 2)` inside the guard?
2. If `shop.prices` is imported, what value does `__name__` have inside `prices.py`?
3. What would happen if the test call in `utils.py` were moved outside the guard?
4. In your own words, what is the difference between a module and a package?

Try answering before looking below.

### Check your answers

1. The `BATCH` assignment runs; the guarded `add(1, 2)` does not.
2. `"shop.prices"`.
3. It would run when `utils` is first imported, as well as when the file is run directly.
4. A module is an importable unit of code; a package groups related modules under a shared import name.

**Remember:** `__name__` is the variable; `"__main__"` is one possible value. The guard compares the two.